#source data

In [0]:
df_src= spark.sql("select * from parquet.`abfss://silver@retailstgaccpd.dfs.core.windows.net/cleaned_data`")

# Getting all the dimention tables

In [0]:
df_branch = spark.sql("select * from cars_data_catalog.gold.dim_branch")
df_dealer = spark.sql("select * from cars_data_catalog.gold.dim_dealer")
df_model = spark.sql("select * from cars_data_catalog.gold.dim_model")
df_date = spark.sql("select * from cars_data_catalog.gold.dim_date")


#Creating Gold Fact Table

In [0]:
df_fact = df_src.join(df_branch, df_src.Branch_ID == df_branch.Branch_ID, "left")\
                .join(df_dealer, df_src.Dealer_ID == df_dealer.Dealer_ID, "left")\
                .join(df_model, df_src.Model_ID == df_model.Model_ID, "left")\
                .join(df_date, df_src.Date_ID == df_date.Date_ID, "left")\
                .select(df_src.Revenue, df_src.Units_Sold, df_src.RevenuePerUnits, df_branch.dim_branch_key, df_dealer.dim_dealer_key, df_model.dim_model_key, df_date.dim_date_key)


Revenue,Units_Sold,RevenuePerUnits,dim_branch_key,dim_dealer_key,dim_model_key,dim_date_key
13363978,2,6681989.0,814,51,195,1001
17376468,3,5792156.0,815,131,261,1001
9664767,3,3221589.0,1,67,148,867
5525304,3,1841768.0,357,52,87,867
12971088,3,4323696.0,683,132,20,69
7321228,1,7321228.0,358,36,242,285
11379294,2,5689647.0,1492,82,160,285
11611234,2,5805617.0,933,119,179,1069
19979446,2,9989723.0,1493,239,262,1069
14181510,3,4727170.0,1493,1,209,1002


# Writing Fact tables to datalake and table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_data_catalog.gold.fact_sales'):
    delta_tbl = DeltaTable.forName(spark, 'cars_data_catalog.gold.fact_sales')
    delta_tbl.alias('trg').merge(df_fact.alias('src'), "trg.dim_model_key = src.dim_model_key and trg.dim_branch_key = src.dim_branch_key and trg.dim_dealer_key =src.dim_dealer_key  and trg.dim_date_key = src.dim_date_key")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    


else:
    df_fact.write.format("delta")\
           .mode("overwrite")\
        .option("path", "abfss://gold@retailstgaccpd.dfs.core.windows.net/fact_sales")\
        .saveAsTable("cars_data_catalog.gold.fact_sales")


In [0]:
%sql
select * from cars_data_catalog.gold.fact_sales

Revenue,Units_Sold,RevenuePerUnits,dim_branch_key,dim_dealer_key,dim_model_key,dim_date_key
13363978,2,6681989.0,814,51,195,1001
17376468,3,5792156.0,815,131,261,1001
9664767,3,3221589.0,1,67,148,867
5525304,3,1841768.0,357,52,87,867
12971088,3,4323696.0,683,132,20,69
7321228,1,7321228.0,358,36,242,285
11379294,2,5689647.0,1492,82,160,285
11611234,2,5805617.0,933,119,179,1069
19979446,2,9989723.0,1493,239,262,1069
14181510,3,4727170.0,1493,1,209,1002
